# 03 — Model Evaluation
Compute ROUGE / BLEU scores on the test split and run qualitative tests.

In [ ]:
!pip install evaluate rouge_score sacrebleu transformers peft datasets --quiet

In [ ]:
import json
import torch
import evaluate
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

BASE_MODEL  = 'google/flan-t5-base'
DATA_DIR    = Path('../backend/data')
MODEL_DIR   = Path('../backend/models')
ADAPTER_DIR = MODEL_DIR / 'lora_adapter'

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load Model & Test Data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
model.to(DEVICE)

dataset = load_from_disk(str(DATA_DIR / 'processed_dataset'))
test_ds = dataset['test']
print(f'Test examples: {len(test_ds)}')

## 2. Generate Predictions

In [ ]:
def generate_answer(question_text, max_new_tokens=300):
    inputs = tokenizer(
        question_text,
        return_tensors='pt',
        max_length=256,
        truncation=True,
    ).to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

predictions, references = [], []

for example in tqdm(test_ds, desc='Generating'):
    pred = generate_answer(example['input_text'])
    predictions.append(pred)
    references.append(example['target_text'])

print(f'Generated {len(predictions)} predictions')

## 3. ROUGE Scores

In [ ]:
rouge = evaluate.load('rouge')
rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)

print('ROUGE Scores:')
for key, val in rouge_scores.items():
    print(f'  {key}: {val:.4f}')

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(rouge_scores.keys(), rouge_scores.values(), color=['#2196F3', '#4CAF50', '#FF9800', '#E91E63'])
ax.set_ylim(0, 1)
ax.set_title('ROUGE Scores — Immigration AI LoRA Model')
ax.set_ylabel('Score')
for bar, val in zip(bars, rouge_scores.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 4. BLEU Score

In [ ]:
bleu = evaluate.load('sacrebleu')
bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in references])
print(f"BLEU Score: {bleu_score['score']:.2f}")

## 5. Qualitative Analysis — Sample Predictions

In [ ]:
results_df = pd.DataFrame({
    'question': [e['input_text'].split('Question: ')[-1].split('\n\nAnswer:')[0] for e in test_ds],
    'reference': references,
    'prediction': predictions,
})

# Show 5 random examples
sample = results_df.sample(5, random_state=42)
for _, row in sample.iterrows():
    print('='*70)
    print(f'Q: {row["question"]}')
    print(f'\nREF: {row["reference"][:300]}...')
    print(f'\nPRED: {row["prediction"][:300]}...')
    print()

## 6. Custom Inference — Test Your Own Questions

In [ ]:
test_questions = [
    'Can I bring my spouse to the US on my H-1B visa?',
    'What is the difference between asylum and refugee status?',
    'How do I renew an expired green card?',
    'What happens if I miss my immigration court hearing?',
    'How long does the naturalization process take?',
]

for q in test_questions:
    prompt = f'Answer the following US immigration question accurately and clearly:\n\nQuestion: {q}\n\nAnswer:'
    answer = generate_answer(prompt)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print('-'*60)

## 7. Save Results

In [ ]:
results_df.to_csv(MODEL_DIR / 'evaluation_results.csv', index=False)
metrics = {'rouge': rouge_scores, 'bleu': bleu_score['score']}
with open(MODEL_DIR / 'evaluation_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Results saved to backend/models/')

## Summary
- Computed ROUGE-1, ROUGE-2, ROUGE-L, and BLEU on test set
- Ran qualitative review on sample predictions
- Results CSV and metrics JSON saved for review
- Proceed to `04_export_onnx.ipynb` to export for fast API inference